In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA silver;
SET TIME ZONE 'UTC';

In [0]:
MERGE INTO silver.characters_history AS target
USING (
  SELECT
    raw_characters_id AS snapshot_id,
    SHA1(CONCAT_WS('|', LOWER(TRIM(character.character.name)), LOWER(TRIM(character.character.world)), source_file_date)) AS snapshot_character_id,
    character.character.name AS name,
    COALESCE(character.character.traded, FALSE) AS traded,
    TO_TIMESTAMP(character.character.deletion_date) AS deletion_at,
    character.character.former_names AS former_names,
    CASE
      WHEN LOWER(TRIM(character.character.title)) = 'none' THEN NULL
      ELSE character.character.title
    END AS title,
    character.character.unlocked_titles AS unlocked_titles,
    character.character.sex AS sex,
    CASE
      WHEN LOWER(TRIM(character.character.vocation)) = 'none' THEN NULL
      ELSE character.character.vocation
    END AS vocation,
    character.character.level AS level,
    character.character.achievement_points AS achievement_points,
    SHA1(LOWER(TRIM(character.character.world))) AS world_id,
    character.character.world AS world,
    character.character.former_worlds AS former_worlds,
    character.character.residence AS residence,
    character.character.married_to AS married_to,
    transform(character.character.houses, h ->
      NAMED_STRUCT(
        'house_id',        cast(h.houseid AS STRING), 
        'name',            h.name,
        'town',            h.town,
        'paid_until_date', to_date(h.paid)
      )
    ) AS houses,
    CASE
      WHEN character.character.guild.name IS NULL AND character.character.guild.rank IS NULL THEN NULL
      ELSE character.character.guild 
    END AS guild,
    TO_TIMESTAMP(character.character.last_login) AS last_login_at,
    character.character.comment AS comment,
    character.character.position AS position,
    character.character.account_status AS account_status,
    character.account_badges AS account_badges,
    character.achievements AS achievements,
    character.deaths_truncated AS deaths_truncated,
    transform(character.deaths, d ->
      NAMED_STRUCT(
        'death_at', TO_TIMESTAMP(d.time),
        'level',    d.level,
        'reason',   d.reason,
        'assists', transform(d.assists, a ->
          NAMED_STRUCT(
            'name',   a.name,
            'player', a.player,
            'traded', a.traded,
            'summon', NULLIF(TRIM(a.summon), '')
          )
        ),
        'killers', transform(d.killers, k ->
          NAMED_STRUCT(
            'name',   k.name,
            'player', k.player,
            'traded', k.traded,
            'summon', NULLIF(TRIM(k.summon), '')
          )
        )
      )
    ) AS deaths,
    CASE 
      WHEN character.account_information.position IS NULL
       AND character.account_information.loyalty_title IS NULL
       AND character.account_information.created IS NULL
      THEN NULL
      ELSE NAMED_STRUCT(
        'position',      character.account_information.position,
        'loyalty_title', character.account_information.loyalty_title,
        'created_at',    TO_TIMESTAMP(character.account_information.created)
      )
    END AS account_information,
    character.other_characters AS other_characters,
    source_file_date,
    ingested_at,
    TO_TIMESTAMP(information.timestamp) AS api_timestamp,
    CURRENT_TIMESTAMP() AS processed_at
  FROM tibia_analytics.bronze.raw_characters
) AS source
ON target.snapshot_id = source.snapshot_id
WHEN NOT MATCHED THEN
  INSERT *;

In [0]:
-- DATA EXCEPTION (upstream API inconsistency)
-- Correcting former_names for character 'Birel Shidinup' due to incorrect historical value
-- This fix is required to ensure identity resolution consistency in downstream pipeline
UPDATE tibia_analytics.silver.characters_history
   SET former_names     = ARRAY('Old Melekawm')
 WHERE snapshot_id      = 'ffc1fca1fa99781df6979f47445f815c54b6d1d0' 
   AND name             = 'Birel Shidinup'
   AND api_timestamp    = '2026-03-17T10:11:54.000+00:00';